# Qwen3-1.7B Clinical Screening JSON Fine-Tuning for `symptom_analysis`

End-to-end notebook that fine-tunes `Qwen/Qwen3-1.7B-Base` with QLoRA via **Unsloth** on the Kaggle dataset `luizaaca/symptoms-to-diseases-with-reasoning` for exclusive use inside the `symptom_analysis` step of the screening workflow.

**Target runtime:** VS Code connected to a remote Google Colab kernel with a Tesla T4 (16 GB VRAM).

This notebook is intentionally scoped:
- it uses the Kaggle dataset as the only training source at runtime;
- it simulates the prompt flow used by `clinical_backend.analyze`;
- it focuses only on fine-tuning, validation, and GGUF artifact generation; and
- it keeps the structured-output contract limited to three dataset-aligned fields.

> ⚠️ This notebook is for research and screening-assistance workflows only. It does not replace professional medical judgment.

## 1. Set Up the Remote Colab Workspace

Install the fine-tuning stack, load the main Python dependencies, confirm Kaggle credentials are available in the remote Colab runtime, verify the GPU, and prepare a Kaggle-only data-loading path.

In [1]:
%%capture
%pip install --upgrade --no-cache-dir unsloth
%pip install --no-deps unsloth_zoo
%pip install bitsandbytes trl peft accelerate datasets kagglehub scikit-learn pandas sentencepiece protobuf pydantic "transformers>=4.51.0" llama-cpp-python

In [ ]:
import gc
import json
import os
import random
import re
import warnings
from pathlib import Path
from typing import Any, cast
from google.colab import userdata # type: ignore
import kagglehub
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from IPython.display import display
from pydantic import BaseModel, Field, ValidationError
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)

CONFIG = {
    "dataset_handle": "luizaaca/symptoms-to-diseases-with-reasoning",
    "base_model_id": "Qwen/Qwen3-1.7B-Base",
    "max_seq_length": 1024,
    "load_in_4bit": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "warmup_steps": 20,
    "max_steps": 400,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "fp16": True,
    "bf16": False,
    "optim": "adamw_8bit",
    "lr_scheduler_type": "linear",
    "test_size": 0.10,
    "eval_max_samples": 300,
    "output_dir": "outputs_qwen3_1_7b_json",
    "lora_dir": "qwen3-1.7b-clinical-json-lora",
    "gguf_dir": "qwen3-1.7b-clinical-json-gguf",
    "valid_support_statuses": ["supported", "inconclusive"],
    "seed": SEED,
}

REQUIRED_COLUMNS = [
    "input",
    "output",
    "support_status",
    "candidate_diseases",
    "recommended_exams_tests",
]

kaggle_username = userdata.get("KAGGLE_USERNAME")
kaggle_key = userdata.get("KAGGLE_KEY")
if not kaggle_username or not kaggle_key:
    print(
        "Kaggle credentials were not found in the remote runtime. "
        "For a Colab kernel, configure KAGGLE_USERNAME and KAGGLE_KEY before downloading the dataset."
    )
else:
    os.environ["KAGGLE_USERNAME"] = kaggle_username
    os.environ["KAGGLE_KEY"] = kaggle_key
    print(f"Kaggle credentials detected for user: {kaggle_username}")

assert torch.cuda.is_available(), "CUDA GPU is required (Google Colab T4 recommended)."
gpu_name = torch.cuda.get_device_name(0)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"Total VRAM: {total_mem_gb:.1f} GB")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print("Configuration loaded.")

Kaggle credentials detected for user: luizaaca
GPU: Tesla T4
Total VRAM: 14.6 GB
PyTorch: 2.10.0+cu128 | CUDA: 12.8
Configuration loaded.


## 2. Download the Kaggle Dataset

Use the Kaggle dataset `luizaaca/symptoms-to-diseases-with-reasoning` as the only training source for this notebook.

In [3]:
def resolve_downloaded_dataset_file(root_dir: str) -> Path:
    """Resolve the single tabular file downloaded from Kaggle.

    Args:
        root_dir: Root directory returned by `kagglehub.dataset_download`.

    Returns:
        Path to the downloaded dataset file.

    Raises:
        FileNotFoundError: If no file is found under the downloaded directory.
        ValueError: If multiple files are found and the dataset layout is ambiguous.
    """

    dataset_files = sorted(path for path in Path(root_dir).rglob("*") if path.is_file())
    if not dataset_files:
        raise FileNotFoundError(f"No dataset file found under {root_dir}")
    if len(dataset_files) != 1:
        raise ValueError(
            f"Expected a single downloaded dataset file under {root_dir}, found {len(dataset_files)} files."
        )
    return dataset_files[0]


dataset_root = kagglehub.dataset_download(CONFIG["dataset_handle"])
DATASET_FILE = resolve_downloaded_dataset_file(dataset_root)

print(f"Dataset root: {dataset_root}")
print(f"Resolved dataset file: {DATASET_FILE.name}")

100%|██████████| 10.9M/10.9M [00:01<00:00, 6.19MB/s]

Extracting files...


Dataset root: /root/.cache/kagglehub/datasets/luizaaca/symptoms-to-diseases-with-reasoning/versions/1
Resolved dataset file: combined_diseases_symptoms_2_enriched_with_exams_v2.csv


## 3. Canonicalize the Specialist Output Contract

The fine-tuned model will learn to emit a single JSON object with exactly three fields:
- `support_status`
- `candidate_diseases`
- `recommended_exams_tests`

This notebook intentionally keeps the structured output limited to the dataset-aligned specialist contract for the `symptom_analysis` step. It does **not** train the full production payload used elsewhere in the application.

During preprocessing, the reference `output` label is guaranteed to appear in `candidate_diseases` before deduplication and serialization if it is not already present.

In [4]:
class ClinicalScreeningOutput(BaseModel):
    """Structured clinical screening payload for the dedicated specialist model.

    Attributes:
        support_status: Free-text support status emitted by the model.
        candidate_diseases: Ordered disease candidates with the primary label first.
        recommended_exams_tests: Ordered confirmatory exams or tests.
    """

    support_status: str = Field(
        description="Support status string. Expected training values are 'supported' or 'inconclusive'.",
    )
    candidate_diseases: list[str] = Field(
        min_length=1,
        description="Ordered candidate diseases with the primary candidate in the first position.",
    )
    recommended_exams_tests: list[str] = Field(
        min_length=1,
        description="Ordered confirmatory exams or tests.",
    )



def clean_text(value: Any) -> str:
    """Return a whitespace-normalized string representation of a value.

    Args:
        value: Arbitrary value to normalize.

    Returns:
        A trimmed string with internal whitespace collapsed to single spaces.
    """

    return re.sub(r"\s+", " ", str(value)).strip()



def normalize_label(value: Any) -> str:
    """Return a normalized disease label for comparisons.

    Args:
        value: Raw disease label.

    Returns:
        Lowercase disease label with collapsed whitespace.
    """

    return clean_text(value).lower()



def unique_preserve(items: list[str]) -> list[str]:
    """Return a list with duplicates removed while preserving the first occurrence.

    Args:
        items: Ordered string values.

    Returns:
        Deduplicated ordered list.
    """

    seen: set[str] = set()
    ordered: list[str] = []
    for item in items:
        cleaned = clean_text(item)
        if not cleaned:
            continue
        key = cleaned.lower()
        if key in seen:
            continue
        seen.add(key)
        ordered.append(cleaned)
    return ordered



def parse_json_list(raw_value: Any, field_name: str) -> list[str]:
    """Parse a JSON-encoded list field from the downloaded dataset.

    Args:
        raw_value: Raw dataset value or already-materialized list.
        field_name: Column name used for clearer error messages.

    Returns:
        Parsed list of strings.

    Raises:
        ValueError: If the value cannot be parsed into a non-string list.
    """

    parsed_value = raw_value
    if isinstance(raw_value, str):
        parsed_value = json.loads(raw_value)
    if not isinstance(parsed_value, list):
        raise ValueError(f"{field_name} must decode to a list.")
    return [clean_text(item) for item in parsed_value if clean_text(item)]



def canonicalize_support_status(raw_value: Any) -> str:
    """Normalize a support-status value without enforcing it for the main benchmark.

    Args:
        raw_value: Raw support-status value.

    Returns:
        Lowercase normalized support status.
    """

    return normalize_label(raw_value)



def canonicalize_output_row(row: dict[str, Any]) -> ClinicalScreeningOutput:
    """Canonicalize one dataset row into the structured target schema.

    Args:
        row: Mapping with dataset columns.

    Returns:
        Canonical structured payload.

    Raises:
        ValueError: If required JSON-list fields are empty after normalization.
    """

    target_label = normalize_label(row["output"])
    candidate_diseases = [
        normalize_label(item) for item in parse_json_list(row["candidate_diseases"], "candidate_diseases")
    ]
    candidate_diseases = unique_preserve([target_label, *candidate_diseases])
    if not candidate_diseases:
        raise ValueError("candidate_diseases cannot be empty after normalization.")

    recommended_exams_tests = unique_preserve(
        parse_json_list(row["recommended_exams_tests"], "recommended_exams_tests")
    )
    if not recommended_exams_tests:
        raise ValueError("recommended_exams_tests cannot be empty after normalization.")

    return ClinicalScreeningOutput(
        support_status=canonicalize_support_status(row["support_status"]),
        candidate_diseases=candidate_diseases,
        recommended_exams_tests=recommended_exams_tests,
    )



def dump_canonical_json(payload: ClinicalScreeningOutput) -> str:
    """Serialize a canonical payload into a stable compact JSON string.

    Args:
        payload: Structured payload to serialize.

    Returns:
        Compact UTF-8 JSON string with stable field order.
    """

    return json.dumps(payload.model_dump(), ensure_ascii=False, separators=(",", ":"))

In [5]:
def stratified_split(
    dataframe: pd.DataFrame,
    test_size: float,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Split a dataframe into train and test partitions with label stratification.

    Args:
        dataframe: Input dataframe with a `normalized_output` column.
        test_size: Test-set ratio.
        seed: Random seed used by `train_test_split`.

    Returns:
        Tuple of `(train_df, test_df)`.
    """

    train_df, test_df = train_test_split(
        dataframe,
        test_size=test_size,
        random_state=seed,
        stratify=dataframe["normalized_output"],
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)



def stratified_eval_sample(dataframe: pd.DataFrame, cap: int) -> pd.DataFrame:
    """Return a capped evaluation subset while preserving class balance as much as possible.

    Args:
        dataframe: Held-out dataframe with a `normalized_output` column.
        cap: Maximum number of rows to keep.

    Returns:
        Stratified evaluation subset.
    """

    if len(dataframe) <= cap:
        return dataframe.reset_index(drop=True)

    shuffled = dataframe.sample(frac=1.0, random_state=SEED)
    per_class = max(1, cap // shuffled["normalized_output"].nunique())
    sampled = shuffled.groupby("normalized_output", group_keys=False).head(per_class)
    if len(sampled) < cap:
        remainder = shuffled.drop(index=sampled.index)
        sampled = pd.concat([sampled, remainder.head(cap - len(sampled))], ignore_index=True)
    elif len(sampled) > cap:
        sampled = sampled.head(cap)
    return sampled.reset_index(drop=True)


raw_df = pd.read_table(DATASET_FILE, sep=",")
missing_columns = [column for column in REQUIRED_COLUMNS if column not in raw_df.columns]
assert not missing_columns, f"Missing required columns: {missing_columns}"

working_df = raw_df.loc[:, REQUIRED_COLUMNS].copy()
canonical_payloads: list[ClinicalScreeningOutput | None] = []
preprocessing_errors: list[str] = []

for row in working_df.itertuples(index=False):
    row_mapping = row._asdict()
    try:
        payload = canonicalize_output_row(cast(dict[str, Any], row_mapping))
        canonical_payloads.append(payload)
        preprocessing_errors.append("")
    except Exception as exc:
        canonical_payloads.append(None)
        preprocessing_errors.append(f"{type(exc).__name__}: {exc}")

valid_mask = np.array([payload is not None for payload in canonical_payloads], dtype=bool)
valid_payloads = [payload for payload in canonical_payloads if payload is not None]
clean_df = working_df.loc[valid_mask].copy().reset_index(drop=True)
clean_df["structured_target"] = pd.Series(
    [payload.model_dump() for payload in valid_payloads],
    dtype="object",
)
clean_df["structured_target_json"] = [dump_canonical_json(payload) for payload in valid_payloads]
clean_df["normalized_output"] = clean_df["output"].map(normalize_label)
clean_df["reference_support_status"] = [payload.support_status for payload in valid_payloads]
clean_df["reference_support_status_valid"] = clean_df["reference_support_status"].isin(CONFIG["valid_support_statuses"])

label_counts = clean_df["normalized_output"].value_counts()
kept_labels = label_counts[label_counts >= 2].index
model_df = clean_df[clean_df["normalized_output"].isin(kept_labels)].reset_index(drop=True)
rare_label_rows_dropped = int(len(clean_df) - len(model_df))

train_df, test_df = stratified_split(model_df, CONFIG["test_size"], SEED)
eval_df = stratified_eval_sample(test_df, CONFIG["eval_max_samples"])

preprocessing_summary = pd.DataFrame(
    [
        {"metric": "raw_rows", "value": int(len(raw_df))},
        {"metric": "rows_after_required_column_selection", "value": int(len(working_df))},
        {"metric": "rows_with_valid_structured_targets", "value": int(len(clean_df))},
        {"metric": "rows_with_preprocessing_errors", "value": int((~valid_mask).sum())},
        {"metric": "rows_dropped_for_rare_labels", "value": rare_label_rows_dropped},
        {"metric": "rows_used_for_training_and_eval", "value": int(len(model_df))},
        {"metric": "unique_normalized_labels", "value": int(model_df["normalized_output"].nunique())},
        {"metric": "eval_rows", "value": int(len(eval_df))},
    ]
)

display(preprocessing_summary)
if (~valid_mask).any():
    invalid_examples = working_df.loc[~valid_mask, ["input", "output"]].head(5).copy()
    invalid_examples["error"] = pd.Series(
        [error for error in preprocessing_errors if error][: len(invalid_examples)],
        dtype="object",
    )
    display(invalid_examples)

print("Preprocessing note: `output` is injected into `candidate_diseases` when missing before deduplication.")

,metric,value
0,raw_rows,248126
1,rows_after_required_column_selection,248126
2,rows_with_valid_structured_targets,248126
3,rows_with_preprocessing_errors,0
4,rows_dropped_for_rare_labels,0
5,rows_used_for_training_and_eval,248126
6,unique_normalized_labels,767
7,eval_rows,300


Preprocessing note: `output` is injected into `candidate_diseases` when missing before deduplication.


## 4. Define the Training Prompt Format

The training messages in this notebook intentionally mimic the current production flow used by `clinical_backend.analyze`:
- the system prompt follows the clinical analysis prompt style from the project;
- the user message follows the `Clinical request` plus `Active patient context` structure; and
- the assistant response is the structured 3-field JSON payload used for specialist fine-tuning.

In [6]:
TRAINING_SYSTEM_PROMPT = """
You are a clinical screening assistant focused on symptom analysis.

Your job is to:
- interpret the user's complaint in context;
- recommend relevant confirmatory exams or next assessment steps;

Output requirements:
- Respond in the same language as the user, eg., English, Spanish, French.
- Produce a structured internal result in JSON format.
- Return only the JSON object.
- Use exactly these keys in the JSON object: support_status, candidate_diseases, recommended_exams_tests.
""".strip()

## 5. Fine-Tune Qwen3-1.7B on Specialist JSON Targets

Load the base model in 4-bit, attach LoRA adapters, render Qwen chat-format examples that mimic the `symptom_analysis` step input, and fine-tune with response-only supervision on the strict 3-field JSON payload.

In [32]:
model, runtime_tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["base_model_id"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model_id"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if not getattr(tokenizer, "chat_template", None):
    fallback_template = getattr(runtime_tokenizer, "chat_template", None)
    if fallback_template:
        tokenizer.chat_template = fallback_template
    else:
        raise RuntimeError(
            "The Hugging Face tokenizer did not expose a chat template. "
            "Update `transformers` and reload the official tokenizer."
        )
tokenizer.padding_side = "right"

model.print_trainable_parameters()
print(
    {
        "tokenizer_class": type(tokenizer).__name__,
        "has_chat_template": bool(tokenizer.chat_template),
        "eos_token": tokenizer.eos_token,
        "pad_token": tokenizer.pad_token,
    }
)

==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-1.7b-base-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030
{'tokenizer_class': 'Qwen2Tokenizer', 'has_chat_template': True, 'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}


In [33]:
def build_active_patient_context_placeholder() -> str:
    """Return the placeholder used when no patient is active.

    Returns:
        Placeholder text aligned with the current project behavior.
    """

    return "No active patient loaded. Analyze based only on the user's description."



def build_training_clinical_input(user_message: str) -> str:
    """Build the human message content used for training and inference.

    Args:
        user_message: Raw complaint or symptom description.

    Returns:
        Prompt text shaped like the current production backend input.
    """

    return (
        "Clinical request:\n"
        f"{clean_text(user_message)}\n\n"
        "Active patient context:\n"
        f"{build_active_patient_context_placeholder()}"
    )



def ensure_qwen_chat_template(tokenizer_instance: Any) -> str:
    """Return the official Qwen chat template from the Hugging Face tokenizer.

    Args:
        tokenizer_instance: Tokenizer used for rendering training and inference chats.

    Returns:
        Chat template string published with the tokenizer.

    Raises:
        RuntimeError: If the tokenizer does not expose a chat template.
    """

    chat_template = getattr(tokenizer_instance, "chat_template", None)
    if not chat_template:
        raise RuntimeError(
            "The official Hugging Face tokenizer did not expose a chat template. "
            "Reload it with `AutoTokenizer.from_pretrained(...)` on a recent transformers release."
        )
    return str(chat_template)



def render_qwen_chat(
    messages: list[dict[str, str]],
    add_generation_prompt: bool = False,
) -> str:
    """Render messages with the official Hugging Face Qwen chat template.

    Args:
        messages: Ordered chat messages with `role` and `content` keys.
        add_generation_prompt: Whether to append an assistant generation marker.

    Returns:
        Chat-formatted prompt string.

    Raises:
        ValueError: If a message has an unsupported role or missing content.
    """

    supported_roles = {"system", "user", "assistant"}
    normalized_messages: list[dict[str, str]] = []
    for message in messages:
        role = clean_text(message.get("role", "")).lower()
        content = str(message.get("content", ""))
        if role not in supported_roles:
            raise ValueError(f"Unsupported chat role: {role!r}")
        if not content.strip():
            raise ValueError(f"Chat message content cannot be empty for role {role!r}.")
        normalized_messages.append({"role": role, "content": content})

    ensure_qwen_chat_template(tokenizer)
    return cast(
        str,
        tokenizer.apply_chat_template(
            normalized_messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        ),
    )



def to_chat_record(row: Any) -> dict[str, Any]:
    """Convert a dataframe row into a Qwen chat record.

    Args:
        row: Dataset row containing input symptoms and canonical target JSON.

    Returns:
        Dictionary with chat messages and the serialized assistant target.
    """

    input_text = clean_text(row["input"])
    target_json = str(row["structured_target_json"])
    messages = [
        {"role": "system", "content": TRAINING_SYSTEM_PROMPT},
        {"role": "user", "content": build_training_clinical_input(input_text)},
        {"role": "assistant", "content": target_json},
    ]
    return {
        "messages": messages,
        "structured_target_json": target_json,
    }



def tokenize_training_example(example: dict[str, Any]) -> dict[str, list[int]]:
    """Tokenize one training example with the official Qwen chat template.

    Args:
        example: Dataset example containing a `messages` list.

    Returns:
        Mapping with tokenized `input_ids` and `attention_mask`.
    """

    messages = cast(list[dict[str, str]], example["messages"])
    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        truncation=True,
        max_length=CONFIG["max_seq_length"],
    )
    return {
        "input_ids": cast(list[int], tokenized["input_ids"]),
        "attention_mask": cast(list[int], tokenized["attention_mask"]),
    }



def build_tokenized_dataset(dataframe: pd.DataFrame) -> Dataset:
    """Render and tokenize a dataframe into a processed training dataset.

    Args:
        dataframe: Training dataframe with canonical JSON targets.

    Returns:
        Hugging Face dataset containing tokenized `input_ids` and `attention_mask`.
    """

    records = [to_chat_record(record) for record in dataframe.to_dict(orient="records")]
    dataset = Dataset.from_list(records)
    dataset = dataset.map(
        tokenize_training_example,
        remove_columns=dataset.column_names,
    )
    return dataset


ensure_qwen_chat_template(tokenizer)
train_ds = build_tokenized_dataset(train_df)
assert len(train_ds) == len(train_df), "Training dataset row count mismatch."
assert train_df["structured_target_json"].str.startswith("{").all(), "All targets must be JSON objects."
assert {"input_ids", "attention_mask"}.issubset(set(train_ds.column_names)), (
    "Training dataset must expose tokenized `input_ids` and `attention_mask`."
)
assert train_ds[0]["input_ids"], "First tokenized example must not be empty."
assert len(train_ds[0]["input_ids"]) <= CONFIG["max_seq_length"], "Tokenized examples must respect max_seq_length."

print(f"Training rows: {len(train_ds):,}")

Map:   0%|          | 0/223313 [00:00<?, ? examples/s]

Training rows: 223,313


In [35]:
import inspect

from transformers import DataCollatorForSeq2Seq



def configure_qwen_tokens(tokenizer_instance: Any) -> tuple[str, str]:
    """Align tokenizer special tokens with the official Qwen training contract.

    Args:
        tokenizer_instance: Tokenizer used by TRL.

    Returns:
        Tuple of `(eos_token, pad_token)` selected for training.

    Raises:
        RuntimeError: If the required Qwen special tokens are unavailable.
    """

    vocabulary = tokenizer_instance.get_vocab()
    if "<|im_end|>" not in vocabulary:
        raise RuntimeError("Qwen turn-ending token '<|im_end|>' was not found in the tokenizer vocabulary.")

    eos_token = "<|im_end|>"
    pad_token = "<|endoftext|>" if "<|endoftext|>" in vocabulary else eos_token
    tokenizer_instance.eos_token = eos_token
    tokenizer_instance.pad_token = pad_token
    ensure_qwen_chat_template(tokenizer_instance)
    return eos_token, pad_token



def build_response_only_dataset(train_dataset: Dataset, tokenizer_instance: Any) -> Dataset:
    """Add labels that train only on assistant responses, without multiprocessing.

    Args:
        train_dataset: Pre-tokenized dataset containing `input_ids`.
        tokenizer_instance: Tokenizer whose chat markers match the dataset.

    Returns:
        Dataset with `labels` added and fully masked rows removed.
    """

    mask_function = cast(
        Any,
        train_on_responses_only(
            None,
            instruction_part="<|im_start|>user\n",
            response_part="<|im_start|>assistant\n",
            tokenizer=tokenizer_instance,
            return_function=True,
            num_proc=1,
        ),
    )

    labeled_dataset = train_dataset.map(mask_function, batched=True)
    before_count = len(labeled_dataset)
    labeled_dataset = labeled_dataset.filter(
        lambda example: any(label != -100 for label in example["labels"])
    )
    removed_count = before_count - len(labeled_dataset)
    if removed_count:
        print(
            f"Removed {removed_count:,} fully masked training rows after response-only labeling."
        )
    return labeled_dataset



def build_sft_config() -> Any:
    """Create a version-compatible SFT configuration for the installed TRL package.

    Returns:
        Configured trainer-native `SFTConfig` instance.

    Raises:
        RuntimeError: If the installed TRL version exposes neither `max_length`
            nor `max_seq_length` in its `SFTConfig` implementation.
    """

    trainer_sft_config_class = cast(Any, SFTTrainer.__init__.__globals__.get("SFTConfig", SFTConfig))
    eos_token, _ = configure_qwen_tokens(tokenizer)
    config_kwargs: dict[str, Any] = {
        "output_dir": CONFIG["output_dir"],
        "per_device_train_batch_size": CONFIG["per_device_train_batch_size"],
        "gradient_accumulation_steps": CONFIG["gradient_accumulation_steps"],
        "warmup_steps": CONFIG["warmup_steps"],
        "max_steps": CONFIG["max_steps"],
        "learning_rate": CONFIG["learning_rate"],
        "weight_decay": CONFIG["weight_decay"],
        "logging_steps": CONFIG["logging_steps"],
        "fp16": CONFIG["fp16"],
        "bf16": CONFIG["bf16"],
        "optim": CONFIG["optim"],
        "lr_scheduler_type": CONFIG["lr_scheduler_type"],
        "seed": SEED,
        "report_to": "none",
    }

    signature = inspect.signature(trainer_sft_config_class.__init__)
    parameters = signature.parameters
    if "max_length" in parameters:
        config_kwargs["max_length"] = CONFIG["max_seq_length"]
    elif "max_seq_length" in parameters:
        config_kwargs["max_seq_length"] = CONFIG["max_seq_length"]
    else:
        raise RuntimeError("Installed TRL version does not expose max_length or max_seq_length in SFTConfig.")

    if "packing" in parameters:
        config_kwargs["packing"] = False
    if "dataset_kwargs" in parameters:
        config_kwargs["dataset_kwargs"] = {"skip_prepare_dataset": True}
    if "dataset_num_proc" in parameters:
        config_kwargs["dataset_num_proc"] = None
    if "eos_token" in parameters:
        config_kwargs["eos_token"] = eos_token

    return trainer_sft_config_class(**config_kwargs)



def build_sft_trainer(
    model_instance: Any,
    tokenizer_instance: Any,
    train_dataset: Dataset,
    sft_config: Any,
) -> SFTTrainer:
    """Create a version-compatible SFT trainer instance.

    Args:
        model_instance: Model to fine-tune.
        tokenizer_instance: Tokenizer or processing object used by the trainer.
        train_dataset: Pre-tokenized training dataset.
        sft_config: Trainer configuration.

    Returns:
        Configured `SFTTrainer` instance.
    """

    trainer_kwargs: dict[str, Any] = {
        "model": model_instance,
        "train_dataset": train_dataset,
        "args": sft_config,
    }

    signature = inspect.signature(SFTTrainer.__init__)
    parameters = signature.parameters
    if "tokenizer" in parameters:
        trainer_kwargs["tokenizer"] = tokenizer_instance
    elif "processing_class" in parameters:
        trainer_kwargs["processing_class"] = tokenizer_instance

    return SFTTrainer(**trainer_kwargs)


sft_config = build_sft_config()
train_ds_labeled = build_response_only_dataset(train_ds, tokenizer)
assert train_ds_labeled[0]["labels"], "Labeled dataset must include non-empty labels."
assert any(label != -100 for label in train_ds_labeled[0]["labels"]), (
    "At least one token in the first labeled sample must contribute to loss."
)
trainer = build_sft_trainer(model, tokenizer, train_ds_labeled, sft_config)
trainer.data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)

print(f"Trainer initialized with {len(train_ds_labeled):,} labeled rows.")

Map:   0%|          | 0/223313 [00:00<?, ? examples/s]

Filter:   0%|          | 0/223313 [00:00<?, ? examples/s]

Trainer initialized with 223,313 labeled rows.


In [36]:
train_result = trainer.train()
print(train_result)

log_history = pd.DataFrame(trainer.state.log_history)
loss_df = log_history.dropna(subset=["loss"])[["step", "loss"]].reset_index(drop=True)
assert len(loss_df) >= 2, "Expected at least two logged loss points during training."
assert loss_df["loss"].iloc[-1] < loss_df["loss"].iloc[0], "Final loss must be lower than initial loss."

display(loss_df.tail(10))

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 223,313 | Num Epochs = 1 | Total steps = 400
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 17,432,576 of 1,738,007,552 (1.00% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
20,1.753949
40,0.918953
60,0.740304
80,0.669086
100,0.658124
120,0.642099
140,0.593856
160,0.619706
180,0.595477
200,0.573866


TrainOutput(global_step=400, training_loss=0.6664958024024963, metrics={'train_runtime': 789.9801, 'train_samples_per_second': 4.051, 'train_steps_per_second': 0.506, 'total_flos': 6000204261408768.0, 'train_loss': 0.6664958024024963, 'epoch': 0.01432959868167692})


,step,loss
10,220,0.554591
11,240,0.592389
12,260,0.563932
13,280,0.561291
14,300,0.539539
15,320,0.548281
16,340,0.580592
17,360,0.550199
18,380,0.537929
19,400,0.535750


## 6. Validate the Fine-Tuned Specialist Model

Run a compact validation pass over the held-out evaluation split, compare the base model against the fine-tuned model, and keep only the metrics directly relevant to structured output quality and label consistency.

In [37]:
from tqdm.auto import tqdm



def cleanup_memory() -> None:
    """Release Python and CUDA memory before large inference steps."""

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



def build_generation_prompt(user_message: str) -> str:
    """Render a single inference prompt for the current training contract.

    Args:
        user_message: User complaint or symptom description.

    Returns:
        Chat-formatted prompt text with a generation marker.
    """

    messages = [
        {"role": "system", "content": TRAINING_SYSTEM_PROMPT},
        {"role": "user", "content": build_training_clinical_input(user_message)},
    ]
    return render_qwen_chat(messages, add_generation_prompt=True)



@torch.inference_mode()
def generate_raw_output(
    mdl: Any,
    user_message: str,
    max_new_tokens: int = 256,
) -> str:
    """Generate one deterministic model response for the supplied request.

    Args:
        mdl: Loaded causal language model.
        user_message: Clinical request text.
        max_new_tokens: Maximum generation length.

    Returns:
        Raw decoded model text.
    """

    messages = [
        {"role": "system", "content": TRAINING_SYSTEM_PROMPT},
        {"role": "user", "content": build_training_clinical_input(user_message)},
    ]
    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=CONFIG["max_seq_length"],
    ).to(mdl.device)
    output_ids = mdl.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0][model_inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()



def extract_json_candidate(raw_text: str) -> tuple[str | None, bool]:
    """Extract the most likely JSON object from a raw model response.

    Args:
        raw_text: Raw generated text.

    Returns:
        Tuple of `(json_text, strict_json)` where `strict_json` is true only when
        the raw stripped response is already a valid JSON object.
    """

    stripped = raw_text.strip()
    if not stripped:
        return None, False

    try:
        parsed = json.loads(stripped)
        if isinstance(parsed, dict):
            return stripped, True
    except json.JSONDecodeError:
        pass

    fenced_match = re.search(r"```(?:json)?\s*(.*?)\s*```", stripped, flags=re.DOTALL | re.IGNORECASE)
    if fenced_match:
        fenced_candidate = fenced_match.group(1).strip()
        try:
            parsed = json.loads(fenced_candidate)
            if isinstance(parsed, dict):
                return fenced_candidate, False
        except json.JSONDecodeError:
            pass

    decoder = json.JSONDecoder()
    for start_index, character in enumerate(stripped):
        if character != "{":
            continue
        try:
            parsed, end_index = decoder.raw_decode(stripped[start_index:])
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            return stripped[start_index : start_index + end_index], False
    return None, False



def parse_prediction(raw_text: str) -> dict[str, Any]:
    """Parse one raw model response into validation-ready structured fields.

    Args:
        raw_text: Raw generated model response.

    Returns:
        Parsed validation dictionary.
    """

    json_candidate, strict_json = extract_json_candidate(raw_text)
    parsed_payload: dict[str, Any] = {}
    json_parsed = False
    schema_valid = False
    support_status = ""
    candidate_diseases: list[str] = []
    recommended_exams_tests: list[str] = []

    if json_candidate is not None:
        loaded_candidate = json.loads(json_candidate)
        if isinstance(loaded_candidate, dict):
            parsed_payload = loaded_candidate
            json_parsed = True
            raw_candidates = loaded_candidate.get("candidate_diseases", [])
            raw_exams = loaded_candidate.get("recommended_exams_tests", [])
            candidate_diseases = unique_preserve(
                [normalize_label(item) for item in raw_candidates] if isinstance(raw_candidates, list) else []
            )
            recommended_exams_tests = unique_preserve(
                [clean_text(item) for item in raw_exams] if isinstance(raw_exams, list) else []
            )
            support_status = canonicalize_support_status(loaded_candidate.get("support_status", ""))
            try:
                ClinicalScreeningOutput(
                    support_status=support_status,
                    candidate_diseases=candidate_diseases,
                    recommended_exams_tests=recommended_exams_tests,
                )
                schema_valid = bool(support_status)
            except ValidationError:
                schema_valid = False

    return {
        "raw_output": raw_text,
        "json_candidate": json_candidate or "",
        "strict_json": strict_json,
        "json_parsed": json_parsed,
        "schema_valid": schema_valid,
        "support_status": support_status,
        "support_status_valid": support_status in CONFIG["valid_support_statuses"],
        "candidate_diseases": candidate_diseases,
        "recommended_exams_tests": recommended_exams_tests,
        "predicted_label": candidate_diseases[0] if candidate_diseases else "",
        "parsed_payload": parsed_payload,
    }



def predict_dataframe(mdl: Any, dataframe: pd.DataFrame, desc: str) -> pd.DataFrame:
    """Run deterministic inference over a dataframe and parse every prediction.

    Args:
        mdl: Loaded model to evaluate.
        dataframe: Evaluation dataframe.
        desc: Progress-bar label.

    Returns:
        Dataframe containing reference columns and parsed predictions.
    """

    rows: list[dict[str, Any]] = []
    for row in tqdm(dataframe.itertuples(index=False), total=len(dataframe), desc=desc):
        raw_output = generate_raw_output(mdl, str(row.input))
        parsed = parse_prediction(raw_output)
        rows.append(
            {
                "input": str(row.input),
                "reference_output": str(row.output),
                "reference_norm": str(row.normalized_output),
                "reference_support_status": str(row.reference_support_status),
                **parsed,
            }
        )
    return pd.DataFrame(rows)



def compute_validation_metrics(predictions_df: pd.DataFrame) -> dict[str, Any]:
    """Compute compact validation metrics for one prediction dataframe.

    Args:
        predictions_df: Parsed prediction dataframe.

    Returns:
        Dictionary with validation metrics.
    """

    y_true = predictions_df["reference_norm"].tolist()
    y_pred = [normalize_label(label) for label in predictions_df["predicted_label"].tolist()]
    target_in_candidates = predictions_df.apply(
        lambda row: row["reference_norm"] in row["candidate_diseases"],
        axis=1,
    )
    predicted_status = [status if status else "<missing>" for status in predictions_df["support_status"].tolist()]
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),
        "strict_json_rate": float(predictions_df["strict_json"].mean()),
        "json_parse_rate": float(predictions_df["json_parsed"].mean()),
        "schema_valid_rate": float(predictions_df["schema_valid"].mean()),
        "target_label_in_candidates_rate": float(target_in_candidates.mean()),
        "non_empty_exam_list_rate": float(predictions_df["recommended_exams_tests"].map(bool).mean()),
        "support_status_accuracy": float(accuracy_score(predictions_df["reference_support_status"].tolist(), predicted_status)),
        "support_status_valid_rate": float(predictions_df["support_status_valid"].mean()),
        "n": int(len(predictions_df)),
    }


FastLanguageModel.for_inference(model)
print("Inference helpers ready.")

Inference helpers ready.


In [38]:
fine_tuned_predictions = predict_dataframe(model, eval_df, "Fine-tuned eval")
with model.disable_adapter():
    base_predictions = predict_dataframe(model, eval_df, "Base eval")

metrics_rows = []
for model_name, predictions_df in [("Base", base_predictions), ("Fine-tuned", fine_tuned_predictions)]:
    metrics = compute_validation_metrics(predictions_df)
    metrics["model"] = model_name
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows)[
    [
        "model",
        "n",
        "accuracy",
        "f1_macro",
        "cohen_kappa",
        "strict_json_rate",
        "json_parse_rate",
        "schema_valid_rate",
        "target_label_in_candidates_rate",
        "non_empty_exam_list_rate",
        "support_status_accuracy",
        "support_status_valid_rate",
    ]
].sort_values("model")

display(metrics_df)

fine_tuned_accuracy = metrics_df.loc[metrics_df["model"] == "Fine-tuned", "accuracy"].iloc[0]
base_accuracy = metrics_df.loc[metrics_df["model"] == "Base", "accuracy"].iloc[0]
print({"fine_tuned_accuracy": fine_tuned_accuracy, "base_accuracy": base_accuracy})
assert fine_tuned_accuracy >= base_accuracy, "Fine-tuned label accuracy should match or exceed the base model."

Fine-tuned eval:   0%|          | 0/300 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Base eval:   0%|          | 0/300 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

,model,n,accuracy,f1_macro,cohen_kappa,strict_json_rate,json_parse_rate,schema_valid_rate,target_label_in_candidates_rate,non_empty_exam_list_rate,support_status_accuracy,support_status_valid_rate
0,Base,300,0.00,0.000000,0.000000,0.0,0.016667,0.01,0.00,0.01,0.000000,0.0
1,Fine-tuned,300,0.21,0.138174,0.207773,0.0,1.000000,1.00,0.28,1.00,0.696667,1.0


{'fine_tuned_accuracy': np.float64(0.21), 'base_accuracy': np.float64(0.0)}


## 7. Export the Adapter and GGUF Artifact

Save the LoRA adapter, export a `q4_k_m` GGUF file, and run a smoke test that verifies the exported model still returns valid JSON for the notebook-local schema.

In [39]:
from datetime import datetime, timezone

try:
    from google.colab import drive  # type: ignore
except ImportError as exc:
    raise RuntimeError(
        "This artifact export path expects a remote Colab runtime with Google Drive available. "
        "If you want a non-Colab export path, create a separate developer-specific variant instead of changing this notebook's default runtime contract."
    ) from exc

drive.mount("/content/drive")
artifact_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
ARTIFACT_ROOT = Path(f"/content/drive/MyDrive/screening_robot/screening_robot_qwen3_1_7b_json/{artifact_stamp}")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
lora_dir = ARTIFACT_ROOT / CONFIG["lora_dir"]
gguf_dir = ARTIFACT_ROOT / CONFIG["gguf_dir"]

model.save_pretrained(str(lora_dir))
tokenizer.save_pretrained(str(lora_dir))
print(f"LoRA adapter saved to: {lora_dir}")

Mounted at /content/drive
LoRA adapter saved to: /content/drive/MyDrive/screening_robot/screening_robot_qwen3_1_7b_json/20260509_170959/qwen3-1.7b-clinical-json-lora


In [41]:
def collect_gguf_files(*candidate_paths: Path) -> list[Path]:
    """Collect GGUF files from known Unsloth export locations.

    Args:
        *candidate_paths: Candidate files or directories to inspect.

    Returns:
        Sorted list of discovered GGUF files.
    """

    discovered: set[Path] = set()
    for candidate in candidate_paths:
        if not candidate.exists():
            continue
        if candidate.is_file() and candidate.suffix.lower() == ".gguf":
            discovered.add(candidate)
            continue
        discovered.update(candidate.rglob("*.gguf"))
    return sorted(discovered)



gguf_dir.mkdir(parents=True, exist_ok=True)
implicit_unsloth_dir = Path(f"{gguf_dir}_gguf")
gguf_files = collect_gguf_files(gguf_dir, implicit_unsloth_dir)
if not gguf_files:
    export_result = model.save_pretrained_gguf(
        str(gguf_dir),
        tokenizer,
        quantization_method="q4_k_m",
    )
    extra_candidates: list[Path] = []
    if isinstance(export_result, (str, Path)):
        extra_candidates.append(Path(export_result))
    elif export_result is not None:
        extra_candidates.extend(Path(item) for item in export_result)

    gguf_files = collect_gguf_files(gguf_dir, implicit_unsloth_dir, *extra_candidates)

assert gguf_files, "Expected at least one GGUF artifact after export."
for gguf_file in gguf_files:
    size_mb = gguf_file.stat().st_size / 1024**2
    print(f"{gguf_file} ({size_mb:.1f} MB)")

/content/drive/MyDrive/screening_robot/screening_robot_qwen3_1_7b_json/20260509_170959/qwen3-1.7b-clinical-json-gguf_gguf/qwen3-1.7b-base.Q4_K_M.gguf (1056.1 MB)


In [44]:
for variable_name in ("trainer", "model"):
    if variable_name in globals():
        del globals()[variable_name]
cleanup_memory()

from llama_cpp import Llama

gguf_path = str(gguf_files[0])
print(f"Loading smoke-test GGUF: {gguf_path}")

llm = Llama(
    model_path=gguf_path,
    n_ctx=2048,
    n_threads=os.cpu_count() or 4,
    n_gpu_layers=-1,
    verbose=False,
)

smoke_prompt = build_generation_prompt(
    "high fever, severe headache, myalgia, rash, nausea"
)
smoke_response = cast(
    Any,
    llm.create_completion(
        prompt=smoke_prompt,
        temperature=0.0,
        max_tokens=256,
        stop=["<|im_end|>", "<|endoftext|>"],
    ),
)
smoke_text = str(smoke_response["choices"][0].get("text", "") or "").strip()
print(smoke_text)

smoke_parsed = parse_prediction(smoke_text)
assert smoke_parsed["json_parsed"], "Smoke-test output must parse as JSON."
assert smoke_parsed["schema_valid"], "Smoke-test output must satisfy the structured schema."
display(pd.DataFrame([smoke_parsed]))
print("Smoke-test contract: the exported model returns only the 3-field specialist payload.")

Loading smoke-test GGUF: /content/drive/MyDrive/screening_robot/screening_robot_qwen3_1_7b_json/20260509_170959/qwen3-1.7b-clinical-json-gguf_gguf/qwen3-1.7b-base.Q4_K_M.gguf


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


paździ

𫟦

{"support_status":"inconclusive","candidate_diseases":["malaria","pneumonia","acute bronchitis","acute pancreatitis"],"recommended_exams_tests":["complete blood count (CBC)","X-ray","comprehensive metabolic panel (CMP)","serum lipase","abdominal ultrasound"]}𫟦
𫟦
𫟦

𫟦

{"support_status":"inconclusive","candidate_diseases":["pneumonia","acute bronchitis","acute pancreatitis"],"recommended_exams_tests":["complete blood count (CBC)","X-ray","comprehensive metabolic panel (CMP)","serum lipase","abdominal ultrasound"]}𫟦
𫟦
𫟦

𫟦

{"support_status":"inconclusive","candidate_diseases":["pneumonia","acute bronchitis"],"recommended_exams_tests":["complete blood count (CBC)","X-ray","comprehensive metabolic panel (CMP)","serum lipase","abdominal ultrasound"]}𫟦
𫟦
ketøy

ketøy

{"support_status":"inconclusive","candidate_diseases":["pneumonia","acute bronchitis"],"recommended_exams_tests":["complete blood count (CBC)","X-ray","comprehensive metabolic panel (CMP)","serum lipase","abdominal 

,raw_output,json_candidate,strict_json,json_parsed,schema_valid,support_status,support_status_valid,candidate_diseases,recommended_exams_tests,predicted_label,parsed_payload
0,"paździ\n\n𫟦\n\n{""support_status"":""inconclusive"",""candidate_diseases"":[""malaria"",""pneumonia"",""acute bronchitis"",""acute pancreatitis""],""recommended_exams_tests"":[""complete blood count (CBC)"",""X-ray""...","{""support_status"":""inconclusive"",""candidate_diseases"":[""malaria"",""pneumonia"",""acute bronchitis"",""acute pancreatitis""],""recommended_exams_tests"":[""complete blood count (CBC)"",""X-ray"",""comprehensive...",False,True,True,inconclusive,True,"[malaria, pneumonia, acute bronchitis, acute pancreatitis]","[complete blood count (CBC), X-ray, comprehensive metabolic panel (CMP), serum lipase, abdominal ultrasound]",malaria,"{'support_status': 'inconclusive', 'candidate_diseases': ['malaria', 'pneumonia', 'acute bronchitis', 'acute pancreatitis'], 'recommended_exams_tests': ['complete blood count (CBC)', 'X-ray', 'com..."


Smoke-test contract: the exported model returns only the 3-field specialist payload.


In [46]:
if "llm" in globals():
    del llm
cleanup_memory()

from llama_cpp import Llama, LlamaGrammar


def build_gguf_response_schema() -> dict[str, Any]:
    """Return the strict JSON Schema used for GGUF smoke-test decoding.

    Returns:
        JSON Schema that constrains the assistant to the 3-field specialist payload.
    """

    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "support_status": {
                "type": "string",
                "enum": list(CONFIG["valid_support_statuses"]),
            },
            "candidate_diseases": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 1,
            },
            "recommended_exams_tests": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 1,
            },
        },
        "required": [
            "support_status",
            "candidate_diseases",
            "recommended_exams_tests",
        ],
    }



def extract_chat_message_content(response: dict[str, Any]) -> str:
    """Return the assistant content from a llama-cpp chat completion response.

    Args:
        response: Raw chat completion response.

    Returns:
        Assistant message content.
    """

    message = cast(dict[str, Any], response["choices"][0]["message"])
    return str(message.get("content", "") or "").strip()


alternate_gguf_path = str(gguf_files[0])
print(f"Loading alternate GGUF smoke-test model: {alternate_gguf_path}")

llm = Llama(
    model_path=alternate_gguf_path,
    n_ctx=2048,
    n_threads=os.cpu_count() or 4,
    n_gpu_layers=-1,
    verbose=False,
    chat_format="chatml",
    seed=SEED,
)

strict_smoke_messages = [
    {"role": "system", "content": TRAINING_SYSTEM_PROMPT},
    {
        "role": "user",
        "content": build_training_clinical_input(
            "high fever, severe headache, myalgia, rash, nausea"
        ),
    },
]
strict_smoke_schema = build_gguf_response_schema()
strict_smoke_kwargs: dict[str, Any] = {
    "messages": strict_smoke_messages,
    "temperature": 0.0,
    "max_tokens": 160,
    "repeat_penalty": 1.1,
}

try:
    strict_smoke_response = cast(
        Any,
        llm.create_chat_completion(
            **strict_smoke_kwargs,
            response_format={
                "type": "json_object",
                "schema": strict_smoke_schema,
            },
        ),
    )
except (TypeError, ValueError):
    strict_smoke_response = cast(
        Any,
        llm.create_chat_completion(
            **strict_smoke_kwargs,
            grammar=LlamaGrammar.from_json_schema(
                json.dumps(strict_smoke_schema, ensure_ascii=False)
            ),
        ),
    )

strict_smoke_finish_reason = str(
    strict_smoke_response["choices"][0].get("finish_reason", "")
)
strict_smoke_text = extract_chat_message_content(
    cast(dict[str, Any], strict_smoke_response)
)
print(strict_smoke_text)
print({"finish_reason": strict_smoke_finish_reason})

strict_smoke_parsed = parse_prediction(strict_smoke_text)
assert strict_smoke_finish_reason != "length", (
    "Alternate smoke-test hit the token limit before finishing. "
    "Increase `max_tokens` only if the schema-constrained payload is being truncated."
)
assert strict_smoke_parsed["strict_json"], "Alternate smoke-test output must be a single JSON object."
assert strict_smoke_parsed["json_parsed"], "Alternate smoke-test output must parse as JSON."
assert strict_smoke_parsed["schema_valid"], "Alternate smoke-test output must satisfy the structured schema."
display(pd.DataFrame([strict_smoke_parsed]))
print("Alternate smoke-test contract: schema-constrained chat completion returned one strict 3-field specialist payload.")

Loading alternate GGUF smoke-test model: /content/drive/MyDrive/screening_robot/screening_robot_qwen3_1_7b_json/20260509_170959/qwen3-1.7b-clinical-json-gguf_gguf/qwen3-1.7b-base.Q4_K_M.gguf


llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


{"support_status":"inconclusive","candidate_diseases":["malaria","pneumonia","acute bronchitis","sepsis"],"recommended_exams_tests":["complete blood count (CBC)","X-ray"]}
{'finish_reason': 'stop'}


,raw_output,json_candidate,strict_json,json_parsed,schema_valid,support_status,support_status_valid,candidate_diseases,recommended_exams_tests,predicted_label,parsed_payload
0,"{""support_status"":""inconclusive"",""candidate_diseases"":[""malaria"",""pneumonia"",""acute bronchitis"",""sepsis""],""recommended_exams_tests"":[""complete blood count (CBC)"",""X-ray""]}","{""support_status"":""inconclusive"",""candidate_diseases"":[""malaria"",""pneumonia"",""acute bronchitis"",""sepsis""],""recommended_exams_tests"":[""complete blood count (CBC)"",""X-ray""]}",True,True,True,inconclusive,True,"[malaria, pneumonia, acute bronchitis, sepsis]","[complete blood count (CBC), X-ray]",malaria,"{'support_status': 'inconclusive', 'candidate_diseases': ['malaria', 'pneumonia', 'acute bronchitis', 'sepsis'], 'recommended_exams_tests': ['complete blood count (CBC)', 'X-ray']}"


Alternate smoke-test contract: schema-constrained chat completion returned one strict 3-field specialist payload.
